# IDF: consistency constraints for the coupling, the models solve themselves

IDF (individual discipline feasible) sits between MDF and SAND. The optimiser owns the
iteration variables **and** the coupling variables -- the copies that open the feedback
loops between models -- and answers one consistency equality per copy next to the file's
own constraints. Solves that live inside a single model are not lifted: the coil width
root find stays its own problem, converged inside every optimiser evaluation. So each
model is internally consistent at every iterate. The models agree with each other only
at the optimum.

IDF takes the same three steps as MDF and differs in the last. Cut the feedback loops.
State the file's problem. Then `IDF()`: the scheme's consistency statements are
`Combine`d into the optimiser, and the statements the models declare themselves are
`Nest`ed inside it. SAND absorbs both groups; MDF nests both. On this file the optimiser
gains three coupling unknowns and three consistency equalities on top of the eight
iteration variables.

In [1]:
import os
import sys
import time
from pathlib import Path

HERE = Path.cwd()                                   # the notebook's own folder
REPO = next(p for p in (HERE, *HERE.parents) if (p / "functional_process").is_dir())
os.chdir(REPO)                                      # input files are named relative to PROCESS/
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import jax

jax.config.update("jax_enable_x64", True)          # PROCESS is float64 throughout
try:                                                # whichever cottax is already on the path
    import cottax
except ImportError:                                 # else the checkout beside this repo
    sys.path.insert(0, str(REPO.parent.parent / "jaxgraph" / "src"))
    import cottax
import numpy as np

print("repo  :", REPO)
print("cottax:", Path(cottax.__file__).parent)

repo  : /home/wrutten/projects/functional_PROCESS/PROCESS
cottax: /home/wrutten/projects/jaxgraph/src/cottax


## The graph

The models of the Helias stellarator input file, as the port declares them. Each node
is one model function; it reads and writes PROCESS's own variables (`.physics.rmajor`
is `data.physics.rmajor`). The file also poses the problem: iteration variables `ixc`,
constraints `icc`, figure of merit. `configurations/stellarator_helias.py` states all
of that as a tree -- the input file converted once -- and `native.reference_of` reads
it without running PROCESS.

In [2]:
CONFIGURATION = "stellarator_helias"                   # configurations/stellarator_helias.py: the machine, its values, its problem

import re

from cottax.interfaces import ExecutableGraph, Plan
from cottax.visualization import problems_at

from functional_process.architecture_examples.notebook_tools import print_recipe
from functional_process.configurations import load
from functional_process.cottax.architectures.evaluate import without_excluded
from functional_process.cottax.input import native
from functional_process.cottax.input.indat import graph_for
from functional_process.cottax.queries import declared
from functional_process.cottax.visualization.grouping import driver_name, problem_kind

configuration = load(CONFIGURATION)
ref = native.reference_of(configuration)      # ixc, icc, bounds, cold values -- PROCESS-free
sw = dict(configuration.problem.switches)     # the static switch values the condition nodes are bound with
machine_graph = graph_for(configuration.machine)
raw = without_excluded(machine_graph)

print(f"{len(raw.nodes)} nodes; cyclic components of sizes {[len(c) for c in raw.graph.cycles]}")
print("problems the models declare themselves:")
for p in declared(raw):
    print(f"   {p.spelling:55s} {problem_kind(raw[p])}")
print(f"\nixc = {ref.ixc}")
print(f"icc = {ref.icc}  ({ref.n_equality} equalities), objective: figure of merit {ref.i_figure_merit}")

152 nodes; cyclic components of sizes [2, 6, 2, 2, 2]
problems the models declare themselves:
   ^problem.stellarator.coils.intersect                    root-find
   ^problem.physics.profiles.ion_vol_avg_temperature       fixed-point
   ^problem.power.delta_eta_step                           fixed-point

ixc = [2, 3, 4, 6, 10, 56, 59, 109]
icc = [2, 16, 24, 8, 17, 18, 67, 82, 83, 62, 32, 34, 35, 65]  (2 equalities), objective: figure of merit 6


## The recipe

### Step 1: cut the feedback loops

Each group of models that feed back into each other is opened with copies. The choice
of which variables to copy is a **scheme**, one graph operation
(`cottax.mdao_architectures`); `mda.SCHEME` is `GaussSeidelMinimal`, the order that
cuts the fewest. Each `FixedPointCut` gives the readers of a variable a copy, `^hat.x`,
and states that the copy equals the computed value -- a fixed-point problem, bound
under `^mda`. The `mda_gauss_seidel` notebook compares the three schemes.

In [3]:
from functional_process.cottax.architectures.mda import SCHEME, cut_graph

plan = Plan(raw) + SCHEME
print_recipe(plan)
for closure in SCHEME.composed(raw):
    print(f"   {closure.problem.spelling}: cut {[c.var.spelling for c in closure.cuts]}")
print("\nsame as mda.cut_graph(raw):", plan.graph == cut_graph(raw))
print("problems the cut minted:", [p.spelling for p in declared(plan.graph) if p not in set(declared(raw))])

   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)

same as mda.cut_graph(raw): True
problems the cut minted: ['^problem.physics.proton_rate_density.cycle', '^problem.fwbs.f_ster_div_single']


### Step 2: state the file's problem

Three kinds of node, all stated the way the models are:

- one **constraint** declaration per active `icc`. Each is two nodes: a body owning
  `.constraints.c<id>` at `.Constraint<id>`, computing what
  `models/constraints.py` computes, and the **requirement** that holds it against zero
  beside it at `^require.Constraint<id>` -- `c = 0` for the equalities and `c <= 0` for
  the inequalities. The requirement is the declaration's own, so a constraint is
  declared where it is computed and nothing states it twice;
- one node for the **figure of merit**, owning `.numerics.objf`;
- the **optimiser**: `Optimise(objf, ixc)` at `.Opt`, with no constraints of its own.

A requirement is asserted and answers nothing, so something has to take it on. That is
the architecture's job, in step 3: it `Combine`s every requirement the design reaches --
the relation joins `.Opt` and the requirement node goes -- and one the design cannot
move is left where it is, for the proof to refuse. `sand.problem_graph` is this one
`to_graph`.

In [4]:
from cottax.interfaces import Insert, Optimise, PathMap, bare_conditions
from cottax.interfaces.pytree_namespace_module import node_and_names

from functional_process.cottax.architectures import sand
from functional_process.cottax.architectures.sand import (
    OPT,
    constraint_declarations,
    iteration_variable_path,
    objective_entry,
)

declarations, equalities, inequalities, omitted = constraint_declarations(
    plan.graph, ref.icc, ref.n_equality, sw)                      # body + requirement, per constraint
objective, objf = objective_entry(plan.graph, ref.i_figure_merit, sw)

design = tuple(iteration_variable_path(i) for i in ref.ixc)
stated = {**declarations, **objective, OPT: Optimise(objf, design)}   # minimise, over the iteration variables
plan = plan + Insert(PathMap(node_and_names(stated)))

requirements = tuple(bare_conditions(plan.graph.definitions))
print(f"{len(requirements)} requirement(s) stated, "
      f"{len(sand.unanswered_requirements(plan.graph))} of them beyond the design's reach")

print_recipe(plan)
if omitted:
    print("constraints this port cannot state yet:", omitted)
reference_graph, _, _ = sand.problem_graph(cut_graph(raw), ref.ixc, ref.icc, ref.n_equality,
                                           ref.i_figure_merit, switch_values=sw)
print("\nsame as sand.problem_graph(...):", plan.graph == reference_graph)

   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)
   insert(.Constraint2, .Constraint16, .Constraint24, .Constraint8, .Constraint17, .Constraint18, .Constraint67,  ...
same as sand.optimise_graph(...): True


Two things are wrong with the graph now, and the proof reports both. Every requirement
is asserted and answers nothing, so no driver can move it. And the optimiser owns the
iteration variables, which almost everything reads, so it closes one big loop over most
of the machine -- a loop that now holds several problems with nothing saying which is
outer. Both are what step 3 decides: which statements the optimiser takes on, and which
are solved inside its iteration. Choosing is choosing the architecture.

In [5]:
from collections import Counter

from cottax.interfaces import violations

big = max(plan.graph.graph.components, key=len)     # the components, with no claim that they can be run
print(f"largest block: {len(big)} of {len(plan.graph.nodes)} nodes, holding",
      [p.spelling for p in declared(plan.graph) if p in set(big)], "\n")

found = list(violations(ExecutableGraph, plan.graph))   # every rule that fails, not just the first
print(Counter(type(v).__name__ for v in found), "\n")
for kind in dict.fromkeys(type(v).__name__ for v in found):   # one example of each
    one = next(v for v in found if type(v).__name__ == kind)
    print(re.sub(r"cycle \(.*?\) declares", "the cycle declares", one.message, flags=re.S), "\n")

largest block: 123 of 170 nodes, holding ['^problem.stellarator.coils.intersect', '^problem.physics.profiles.ion_vol_avg_temperature', '^problem.physics.proton_rate_density.cycle', '^problem.fwbs.f_ster_div_single', '.Opt'] 



the block declares several problems ((NodePath(^problem.stellarator.coils.intersect), NodePath(^problem.physics.profiles.ion_vol_avg_temperature), NodePath(^problem.physics.proton_rate_density.cycle), NodePath(^problem.fwbs.f_ster_div_single), NodePath(.Opt))) with nothing saying which is outer -- one driver answers one problem, so `Combine` them into a single problem over every unknown, or `NestInside` one of them. Which is a modelling decision, not something the blocking can read off the graph


### Step 3: absorb the coupling, nest the rest

`IDF()` is the architecture as one operation. It splits the problems on the optimiser's
cycle by where they came from: a statement the **scheme** minted (`^mda`) is the
coupling between models and is `Combine`d into the optimiser -- its unknowns become the
optimiser's unknowns and its relations the optimiser's equalities -- while a statement a
**model** declares itself (`^problem`) is internal to that model and is `Nest`ed inside
the optimiser, converged at every iterate. Membership is read off the namespace, never
stated.

In [6]:
from cottax.mdao_architectures import IDF

from functional_process.cottax.architectures import idf

architecture = IDF(optimiser=OPT)
before = set(declared(plan.graph))
plan = plan + architecture
graph = plan.graph
print("the whole recipe:")
print_recipe(plan)

print("\nabsorbed into the optimiser:", [p.spelling for p in before - set(declared(graph))])
print("nested inside it:           ", {i.spelling: o.spelling for i, o in graph.within.items()})
print("answered at the top level:", [p.spelling for p in problems_at(graph) if p is not None])
print(f"the optimiser now owns {len(graph[OPT].unknowns)} unknowns and reads {len(graph[OPT].conditions)} conditions")

reference_idf, _, _ = idf.idf_graph(machine_graph, ref.ixc, ref.icc, ref.n_equality,
                                    ref.i_figure_merit, switch_values=sw)
print("\nsame as idf.idf_graph(...):", graph == reference_idf)

coupling (folded into the optimiser): ['^problem.physics.proton_rate_density.cycle', '^problem.fwbs.f_ster_div_single']
internal (nested inside it):         ['^problem.stellarator.coils.intersect', '^problem.physics.profiles.ion_vol_avg_temperature', '^problem.power.delta_eta_step'] 

the whole recipe:
   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)
   insert(.Constraint2, .Constraint16, .Constraint24, .Constraint8, .Constraint17, .Constraint18, .Constraint67,  ...
   residualise(^problem.physics.proton_rate_density.cycle)
   residualise(^problem.fwbs.f_ster_div_single)
   combine(^problem.idf <- .Opt, ^problem.physics.proton_rate_density.cycle, ^problem.fwbs.f_ster_div_single)
   nest_inside(^problem.idf)

answered at the top level: ['^problem.idf', '^problem.power.delta_eta_step']
nested inside the optimiser's

### Assign the drivers

VMCON on the optimiser. Newton on the nested root find; it converges inside every VMCON
evaluation and is differentiated through.

In [7]:
from functional_process.cottax.architectures.sand import sand_schedule, sand_shape

schedule = sand_schedule(graph, None, bounds=ref.bounds)
shape = sand_shape(schedule)
drive = shape["drive"]
runnable = schedule.executable.graph
for p in declared(runnable):
    print(f"{p.spelling:55s} {problem_kind(runnable[p]):12s} {driver_name(runnable[p])}")
print(f"\nthe IDF block: {shape['drive_nodes']} nodes, {shape['unknowns']} unknowns "
      f"({shape['design']} design), {shape['conditions']} conditions "
      f"({shape['equalities']} equalities, {shape['inequalities']} inequalities); "
      f"{shape['schedule_steps']} schedule steps in all")
print("unknowns:", [u.spelling for u in drive.unknowns])

the IDF block: 123 nodes, 11 unknowns (11 design), 18 conditions (5 equalities, 12 inequalities); 47 schedule steps in all
unknowns: ['.physics.b_plasma_toroidal_on_axis', '.physics.rmajor', '.physics.temp_plasma_electron_vol_avg_kev', '.physics.nd_plasma_electrons_vol_avg', '.physics.hfact', '.tfcoil.t_tf_superconductor_quench', '.tfcoil.f_a_tf_turn_cable_copper', '.physics.f_nd_alpha_thermal_electron', '^hat.physics.proton_rate_density', '^hat.physics.fusden_alpha_total', '^hat.fwbs.f_ster_div_single']
   ^problem.stellarator.coils.intersect                    root-find    SeededNewtonDriver
   ^problem.physics.profiles.ion_vol_avg_temperature       fixed-point  PicardDriver
   ^problem.power.delta_eta_step                           fixed-point  PicardDriver
   ^problem.idf                                            combined     VmconDriver


## The process

The DSM in run order. The optimiser's box surrounds the coupled block, and inside it are
the smaller boxes of the model-internal solves. MDF shows a box around every iterated
group and SAND shows none inside; IDF shows exactly the model-internal ones. The DSM is written as an interactive page next to this notebook. In the page, hover a
cell for the variables it carries and click a box to fold it.

In [8]:
from functional_process.cottax.visualization.grouping import (
    render_grouped_dsm_html,
    structure_order,
)
from functional_process.cottax.visualization.render_xdsm import SPELLING

drawn = schedule.executable
dsm = render_grouped_dsm_html(
    drawn, order=structure_order(drawn),
    title="stellarator_helias -- IDF: coupling lifted into VMCON, discipline solves nested inside",
    file_name="dsm_idf", outdir=str(HERE), write=True, formatter=SPELLING,
)
print("written:", dsm.path)

Using adapted ragraph from debug branch


written: /home/wrutten/projects/functional_PROCESS/PROCESS/functional_process/architecture_examples/idf/dsm_idf.html


## Run it

As `session.solve_block` does it -- and identically for MDF, IDF and SAND, so the three
arms differ in the architecture and in nothing else. The driver records each iterate and
scales each coupling condition by the size of its quantity, so a condition on a value of
order `1e17` is brought to order one. The block starts with the iteration variables at
the input file's values and everything else at a converged analysis of that design. The
schedule then runs step by step, because VMCON runs outside the compiled program.

In [9]:
from functional_process.cottax.architectures import mdf
from functional_process.cottax.architectures.drivers import Status
from functional_process.cottax.architectures.evaluate import (
    inputs_only,
    mda_env,
    run_schedule,
    seed_block,
)
from functional_process.cottax.architectures.mda import seed_starts
from functional_process.cottax.architectures.sand import residual_condition_scales
from functional_process.cottax.architectures.session import (
    SAND_MAX_ITER,
    recorder,
    trace_tail,
)

_driven, env = mda_env(ref, graph=machine_graph)         # one converged MDA on the cut graph, to seed and scale from

trace = []
solve = sand_schedule(graph, None, bounds=ref.bounds,
                      condition_scale=residual_condition_scales(drive, env),
                      callback=recorder(trace), max_iter=SAND_MAX_ITER)
solve_drive = sand_shape(solve)["drive"]
design_vars = set(design)
seeded, borrowed = seed_block(solve, solve_drive, ref.cold, env, design=design_vars)
starts = seed_starts(solve, env, exclude=design_vars)     # the nested solves' own start ports
seeded.update(starts)
print("start ports re-seeded from the MDA:", [v.spelling for v in starts], "\n")
began = time.perf_counter()
out = run_schedule(solve, inputs_only(solve, seeded), whole=False)
print(f"solved in {time.perf_counter() - began:.1f} s (first solve: includes compilation)")

status = int(np.asarray(mdf.verdict(out, Status, solve_drive)))
iterations, objf, max_eq, min_ie = trace_tail(trace)
print(f"VMCON: status {status}, {iterations} iterations")
print(f"objective {objf:.8f}   max|eq| {max_eq:.1e}   min ineq {min_ie:+.1e}")
print("design:", {i: round(float(np.asarray(out[iteration_variable_path(i)])), 4) for i in ref.ixc})

start ports re-seeded from the MDA: ['^guess.stellarator.wp_width_r_min', '^guess.physics.temp_plasma_ion_vol_avg_kev', '^guess.power.delta_eta', '^guess.physics.proton_rate_density', '^guess.physics.fusden_alpha_total', '^guess.fwbs.f_ster_div_single'] 



solved in 9.5 s (first solve: includes compilation)
VMCON: status 0, 23 iterations
objective 1.21844143   max|eq| 3.6e-11   min ineq -7.8e-10
design: {2: 4.7164, 3: 26.6445, 4: 5.7027, 6: 1.7391773832251176e+20, 10: 1.048, 56: 31.8104, 59: 0.7177, 109: 0.0299}



warm solve: 0.62 s, 23 iterations


## The three answers to one loop

| | MDF | IDF | SAND |
|---|---|---|---|
| the last op | `MDF()` -- nest everything inside the optimiser | `IDF()` -- absorb the coupling, nest the model-internal solves | `SAND()` -- absorb everything |
| optimiser unknowns | iteration variables | + coupling copies | + coupling copies + every model-internal unknown |
| extra equalities | none | one per coupling copy | one per absorbed unknown |
| per evaluation | a converged analysis | one pass, model-internal solves converged | one pass |
| models consistent | at every iterate | within each model at every iterate; between models at the optimum | at the optimum |

Same models, same optimum. Only the last operation differs, and the graph records which
was applied.

In [10]:
RESULT = {"status": status, "iterations": iterations, "objf": objf, "max_eq": max_eq, "min_ie": min_ie,
          "design": {i: float(np.asarray(out[iteration_variable_path(i)])) for i in ref.ixc},
          "unknowns": len(solve_drive.unknowns), "conditions": len(solve_drive.conditions),
          "nested": [p.spelling for p in problems_at(graph.interior(OPT)) if p is not None]}
RESULT

{'status': 0,
 'iterations': 23,
 'objf': 1.218441432809987,
 'max_eq': 3.639666346089143e-11,
 'min_ie': -7.814808800077344e-10,
 'design': {2: 4.7164495986924,
  3: 26.644455788358055,
  4: 5.702734495286721,
  6: 1.7391773832251176e+20,
  10: 1.047952388067319,
  56: 31.81044366522202,
  59: 0.7176937077117317,
  109: 0.02992503718934735},
 'unknowns': 11,
 'conditions': 18,
 'nested': ['^problem.physics.profiles.ion_vol_avg_temperature',
  '^problem.stellarator.coils.intersect']}